# Denoising Autoencoder Detector

### Import module

In [ ]:
import pandas
import yaml
import torch
import raha
import pickle

from modules.Preprocessing import Preprocessing
from modules.Trainer import DAETrainer
from modules.Detector import Detector

from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

### Load Dataset

In [ ]:
yaml_path = "dicts_full.yaml"
datasets = {}

with open(yaml_path, "rb") as file:
  dictionaries = yaml.safe_load(file)

for name, dict in dictionaries.items():
  datasets[name] = {
    "raw_clean": pandas.read_csv(dict["clean_path"]),
    "raw_dirty": pandas.read_csv(dict["path"]),
  }
  
  print(f"{name} → clean: {datasets[name]['raw_clean'].shape}, dirty: {datasets[name]['raw_dirty'].shape}")

### Preprocessing

In [ ]:
for name, df_dict in datasets.items():
  df_clean = df_dict["raw_clean"].copy()
  df_dirty = df_dict["raw_dirty"].copy()

  num_cols = df_clean.select_dtypes(include="number").columns.tolist()
  cat_cols = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()

  prep = Preprocessing(num_cols, cat_cols)
  df_clean = prep.fill_missing(df_clean)
  X_transformed = prep.fit_transform(df_clean)
  X_tensor = torch.tensor(X_transformed)
  loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)

  datasets[name] = {
    "df_clean": df_clean,
    "df_dirty": df_dirty,
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "prep": prep,
    "X_tensor": X_tensor,
    "loader": loader,
  }

  print(f"\n{name}")
  print(f"Total dim after preprocessing: {datasets[name]['prep'].total_dim}")
  print(f"Num cols: {datasets[name]['num_cols']}")
  print(f"Cat cols: {datasets[name]['cat_cols']}")

### Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for name, ds in datasets.items():
  print(f"Training: {name}")

  trainer = DAETrainer(
    prep = ds["prep"],
    device = device,
  )

  df_history = trainer.fit(ds["loader"])
  datasets[name]["trainer"] = trainer
  datasets[name]["df_history"] = df_history
  

### DAE Detector

In [ ]:
detectors = {}

for name, ds in datasets.items():
  print(f"Detecting: {name}")

  detector = Detector(
    trainer = ds["trainer"],
    prep = ds["prep"],
  )

  detector.fit_clean(ds["X_tensor"])
  detector.detect(ds["df_dirty"], ds["df_clean"])
  detector.summary()
  
  detectors[name] = detector

### Output for Baran (Corrector)

In [ ]:
dae_result_path = "./results/detection/DAE"

for name, dict in dictionaries.items():
  d = raha.dataset.Dataset(dict)
  d.detected_cells = detectors[name].get_detected_cells(datasets[name]["df_dirty"])

  # save output
  with open(f"{dae_result_path}/{name}.pkl", "wb") as f:
    pickle.dump(d, f)

  print(f"{name}: {len(d.detected_cells)} detected cells saved")